**PP434 portfolio** · Jakob Kilian Bilan · CC6, part 1

# Downloading six series with a loop

Six indicators for England's strategic authorities from the PP434 course data (week 7). Each series is downloaded in a loop and saved as its own JSON file in `data/devolution/`.

In [ ]:
import io, json, os, time
import requests
import pandas as pd

BASE = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week07/data/msa/"

# One entry per series: file name in the course data, plus a title and unit for the dashboard
SERIES = {
    "gva_per_head":         ("gva_per_head_MSA_ENG.csv",         "Economic output (GVA) per resident", "£, current prices"),
    "pay_fulltime":         ("pay_fulltime_MSA_ENG.csv",         "Median full-time pay (residents)",   "£ a year"),
    "employment_rate":      ("employment_rate_MSA_ENG.csv",      "Employment rate, age 16-64",          "%"),
    "life_expectancy_male": ("life_expectancy_male_MSA_ENG.csv", "Male life expectancy at birth",       "years (3-year average)"),
    "old_age_dependency":   ("old_age_dependency_MSA_ENG.csv",   "People 65+ per 100 aged 16-64",       "ratio"),
    "emissions_per_head":   ("emissions_per_head_MSA_ENG.csv",   "Greenhouse gas emissions per resident", "tonnes CO2e"),
}

OUT = "data/devolution"
os.makedirs(OUT, exist_ok=True)
OUTPUTS = [f"{OUT}/{key}.json" for key in SERIES]

In [ ]:
names = pd.read_csv(BASE + "msa_latest.csv")[["geo_id", "name", "short"]]

for key, (file, title, unit) in SERIES.items():
    url = BASE + file
    response = requests.get(url, timeout=30)
    print(f"{key:22s} HTTP {response.status_code}  {url}")
    if response.status_code != 200:
        print("   -> nothing saved")
        continue
    df = pd.read_csv(io.StringIO(response.text))
    # '2001-2003' (3-year periods) becomes 2003; '1998' stays 1998
    df["year"] = df["date"].astype(str).str[-4:].astype(int)
    df = df.merge(names, on="geo_id", how="left")
    missing = df["name"].isna().sum()
    record = {
        "series": key, "title": title, "unit": unit,
        "source": "ONS and others, compiled by the PP434 course team (week 7 data)",
        "source_url": url,
        "data": df[["geo_id", "name", "short", "date", "year", "value"]].to_dict("records"),
    }
    with open(f"{OUT}/{key}.json", "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False)
    print(f"   -> {len(df)} rows, {df['geo_id'].nunique()} areas, years {df['year'].min()}-{df['year'].max()}, unnamed rows: {missing}")
    time.sleep(0.5)   # be polite to the server

# Also keep one tidy (long) table of all six series, for the interactive explorer in CC9
frames = []
for key in SERIES:
    with open(f"{OUT}/{key}.json", encoding="utf-8") as f:
        d = json.load(f)
    part = pd.DataFrame(d["data"])
    part.insert(0, "indicator", key)
    frames.append(part)
long = pd.concat(frames, ignore_index=True)[["indicator", "geo_id", "name", "short", "date", "year", "value"]]
long.to_csv("data/cc9_strategic_authorities.csv", index=False)
print("tidy table:", long.shape, "->", "data/cc9_strategic_authorities.csv")

## Also: one tidy table of all six series (used by the interactive explorer in CC9)

In [ ]:
frames = []
for key in SERIES:
    with open(f"{OUT}/{key}.json", encoding="utf-8") as f:
        d = json.load(f)
    part = pd.DataFrame(d["data"])
    part.insert(0, "indicator", key)
    frames.append(part)
long = pd.concat(frames, ignore_index=True)[["indicator", "geo_id", "name", "short", "date", "year", "value"]]
long.to_csv("data/cc9_strategic_authorities.csv", index=False)
print("tidy table:", long.shape, "->", "data/cc9_strategic_authorities.csv")
OUTPUTS.append("data/cc9_strategic_authorities.csv")

In [ ]:
# Check: six files, each with the strategic authorities and a sensible range of years
import glob
for f in sorted(glob.glob(f"{OUT}/*.json")):
    with open(f, encoding="utf-8") as fh:
        d = json.load(fh)
    years = [r["year"] for r in d["data"]]
    print(f"{f:40s} {len(d['data']):4d} rows  {min(years)}-{max(years)}  {d['title']}")

In [ ]:
# Colab only: download the output to your computer, then upload it to the data folder of your GitHub repository
try:
    from google.colab import files
    for f in OUTPUTS:
        files.download(f)
except ImportError:
    print("Not running in Colab: the files are in the working folder:", OUTPUTS)